In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed2904/asdfg12345/Malware_Dataset.csv


In [7]:
# ================================================================
# CELL 1: DEEPSEEK-CODER MALWARE DATASET PREPARATION
# CSV -> VALIDATION -> SPLITTING -> PROMPTS -> MASKED TOKENIZATION
# ================================================================

import os
import glob
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
MAX_LENGTH = 4096

WORK_DIR = "/kaggle/working/deepseek_malware"

os.makedirs(WORK_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Configuration initialized.")


# ================================================================
# 2. AUTOMATICALLY LOCATE AND LOAD CSV
# ================================================================

csv_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

valid_files = []

for path in csv_files:
    try:
        columns = pd.read_csv(path, nrows=0).columns

        if {"SHA256", "Type"}.issubset(columns):
            valid_files.append(path)

    except Exception:
        continue

if not valid_files:
    raise FileNotFoundError(
        "No CSV containing SHA256 and Type was found."
    )

valid_files.sort(key=os.path.getsize, reverse=True)

CSV_PATH = valid_files[0]

print("\nSelected CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)


# ================================================================
# 3. DATASET VALIDATION
# ================================================================

assert "SHA256" in df.columns
assert "Type" in df.columns

assert df["SHA256"].is_unique, "Duplicate SHA256 detected."

assert df.isnull().sum().sum() == 0, "Missing values detected."

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

assert len(FEATURE_COLUMNS) == 318, (
    f"Expected 318 features, found {len(FEATURE_COLUMNS)}"
)

assert all(
    pd.api.types.is_numeric_dtype(df[col])
    for col in FEATURE_COLUMNS
), "Non-numeric feature detected."

df["Type"] = pd.to_numeric(
    df["Type"],
    errors="raise"
).astype(int)

LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

assert set(df["Type"].unique()) == set(LABEL_MAP.keys())

print("\nDataset validation successful.")

print("Total samples:", len(df))
print("Total PE features:", len(FEATURE_COLUMNS))

print("\nClass distribution:")
print(df["Type"].value_counts().sort_index())


# ================================================================
# 4. FEATURE GROUPS
# ================================================================

API_FEATURES = FEATURE_COLUMNS[:200]
DLL_FEATURES = FEATURE_COLUMNS[200:227]
HEADER_FEATURES = FEATURE_COLUMNS[227:273]
SECTION_FEATURES = FEATURE_COLUMNS[273:318]

assert len(API_FEATURES) == 200
assert len(DLL_FEATURES) == 27
assert len(HEADER_FEATURES) == 46
assert len(SECTION_FEATURES) == 45

print("\nFeature groups:")
print("API Functions:", len(API_FEATURES))
print("DLL Imports:", len(DLL_FEATURES))
print("PE Header:", len(HEADER_FEATURES))
print("PE Sections:", len(SECTION_FEATURES))


# ================================================================
# 5. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ================================================================

development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["Type"]
)

train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=SEED,
    stratify=development_df["Type"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert len(train_df) == 17693
assert len(validation_df) == 5898
assert len(test_df) == 5898

# Verify no sample leakage.

assert set(train_df["SHA256"]).isdisjoint(
    validation_df["SHA256"]
)

assert set(train_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

assert set(validation_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

print("\nDataset split completed.")

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Testing:", len(test_df))

# Save splits.

for name, split_df in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:

    split_df.to_csv(
        f"{WORK_DIR}/{name}.csv",
        index=False
    )


# ================================================================
# 6. LOAD DEEPSEEK TOKENIZER
# ================================================================

print("\nLoading DeepSeek tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)


# ================================================================
# 7. PROMPT TEMPLATE
# ================================================================

SYSTEM_PROMPT = """You are a Windows PE malware classification assistant.

Classify the executable using only its provided static PE features.

Instructions:
1. Analyze all provided feature values.
2. Zero values are meaningful and must not be ignored.
3. Do not invent missing information.
4. Do not infer unsupported runtime behavior.
5. Select exactly one malware family.

Available classes:
Benign, RedLineStealer, Downloader, RAT,
BankingTrojan, SnakeKeyLogger, Spyware.
"""


def format_features(row, columns):

    return "\n".join(
        f"{col}={row[col]}"
        for col in columns
    )


def create_prompt(row):

    return f"""### SYSTEM:
{SYSTEM_PROMPT}

### USER:
Analyze the following Windows PE static feature profile.

<API_FUNCTIONS>
{format_features(row, API_FEATURES)}
</API_FUNCTIONS>

<DLL_IMPORTS>
{format_features(row, DLL_FEATURES)}
</DLL_IMPORTS>

<PE_HEADER>
{format_features(row, HEADER_FEATURES)}
</PE_HEADER>

<PE_SECTIONS>
{format_features(row, SECTION_FEATURES)}
</PE_SECTIONS>

Classify this executable into exactly one of the seven malware families.

### ASSISTANT:
"""


def create_completion(row):

    return (
        "Predicted Malware Family: "
        + LABEL_MAP[int(row["Type"])]
    )


# ================================================================
# 8. MASKED SFT TOKENIZATION
# ================================================================

def prepare_records(dataframe, split_name):

    records = []

    max_length_observed = 0

    for index, row in dataframe.iterrows():

        prompt = create_prompt(row)

        completion = create_completion(row)

        # Tokenize prompt separately.

        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]

        # Tokenize assistant response separately.

        completion_ids = tokenizer(
            completion,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]

        # Add supervised EOS token.

        completion_ids.append(tokenizer.eos_token_id)

        input_ids = prompt_ids + completion_ids

        # Mask prompt tokens.

        labels = (
            [-100] * len(prompt_ids)
            + completion_ids
        )

        if len(input_ids) > MAX_LENGTH:

            raise ValueError(
                f"{split_name} sample {index} exceeds "
                f"{MAX_LENGTH} tokens: {len(input_ids)}"
            )

        assert all(
            x == -100 for x in labels[:len(prompt_ids)]
        )

        assert labels[len(prompt_ids):] == completion_ids

        assert input_ids[-1] == tokenizer.eos_token_id

        records.append({
            "input_ids": input_ids,
            "labels": labels,
            "label_id": int(row["Type"]),
            "sha256": str(row["SHA256"])
        })

        max_length_observed = max(
            max_length_observed,
            len(input_ids)
        )

        if (index + 1) % 2000 == 0:

            print(
                f"{split_name}: "
                f"{index + 1}/{len(dataframe)} processed"
            )

    print(
        f"{split_name} completed | "
        f"Maximum length: {max_length_observed}"
    )

    return records


print("\nStarting tokenization...")

train_records = prepare_records(
    train_df, "Training"
)

validation_records = prepare_records(
    validation_df, "Validation"
)

test_records = prepare_records(
    test_df, "Testing"
)


# ================================================================
# 9. SAVE TOKENIZED DATASETS
# ================================================================

for name, records in [
    ("train", train_records),
    ("validation", validation_records),
    ("test", test_records)
]:

    output_path = f"{WORK_DIR}/{name}_tokenized.jsonl"

    with open(output_path, "w", encoding="utf-8") as f:

        for record in records:

            f.write(
                json.dumps(record) + "\n"
            )

    print(f"Saved: {output_path}")


# ================================================================
# 10. FINAL VERIFICATION
# ================================================================

print("\n" + "=" * 65)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("\nDataset sizes:")
print("Training:", len(train_records))
print("Validation:", len(validation_records))
print("Testing:", len(test_records))

print("\nMaximum sequence lengths:")

for name, records in [
    ("Training", train_records),
    ("Validation", validation_records),
    ("Testing", test_records)
]:

    print(
        name,
        max(len(x["input_ids"]) for x in records)
    )

sample = train_records[0]

masked_tokens = sum(
    x == -100 for x in sample["labels"]
)

supervised_tokens = sum(
    x != -100 for x in sample["labels"]
)

print("\nMasked SFT verification:")
print("Masked prompt tokens:", masked_tokens)
print("Supervised response tokens:", supervised_tokens)

print("\nDecoded assistant response:")

print(
    tokenizer.decode(
        sample["input_ids"][masked_tokens:],
        skip_special_tokens=False
    )
)

print("\nAll files saved in:")
print(WORK_DIR)

print("\nREADY FOR CELL 2: DEEPSEEK MODEL + LoRA SETUP")

Configuration initialized.

Selected CSV: /kaggle/input/datasets/mohdtahasayed2904/asdfg12345/Malware_Dataset.csv
Dataset shape: (29489, 320)

Dataset validation successful.
Total samples: 29489
Total PE features: 318

Class distribution:
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4217
6    3699
Name: count, dtype: int64

Feature groups:
API Functions: 200
DLL Imports: 27
PE Header: 46
PE Sections: 45

Dataset split completed.
Training: 17693
Validation: 5898
Testing: 5898

Loading DeepSeek tokenizer...


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

Tokenizer loaded.
EOS token: <｜end▁of▁sentence｜>
EOS ID: 32014

Starting tokenization...
Training: 2000/17693 processed
Training: 4000/17693 processed
Training: 6000/17693 processed
Training: 8000/17693 processed
Training: 10000/17693 processed
Training: 12000/17693 processed
Training: 14000/17693 processed
Training: 16000/17693 processed
Training completed | Maximum length: 2963
Validation: 2000/5898 processed
Validation: 4000/5898 processed
Validation completed | Maximum length: 2958
Testing: 2000/5898 processed
Testing: 4000/5898 processed
Testing completed | Maximum length: 2963
Saved: /kaggle/working/deepseek_malware/train_tokenized.jsonl
Saved: /kaggle/working/deepseek_malware/validation_tokenized.jsonl
Saved: /kaggle/working/deepseek_malware/test_tokenized.jsonl

CELL 1 COMPLETED SUCCESSFULLY

Dataset sizes:
Training: 17693
Validation: 5898
Testing: 5898

Maximum sequence lengths:
Training 2963
Validation 2958
Testing 2963

Masked SFT verification:
Masked prompt tokens: 2853
Sup

In [8]:
# ================================================================
# CELL 2: DEEPSEEK MODEL + QLoRA + PYTORCH DATA PIPELINE
# ================================================================

import os
import gc
import json
import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# ================================================================
# 1. CONFIGURATION
# ================================================================

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

WORK_DIR = "/kaggle/working/deepseek_malware"

MODEL_OUTPUT = "/kaggle/working/deepseek_malware_final"

MAX_LENGTH = 4096

MICRO_BATCH_SIZE = 1

os.makedirs(MODEL_OUTPUT, exist_ok=True)

assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))
print("Available GPUs:", torch.cuda.device_count())


# ================================================================
# 2. LOAD TOKENIZER
# ================================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("\nTokenizer loaded.")


# ================================================================
# 3. LOAD TOKENIZED JSONL DATA
# ================================================================

def load_records(filename):

    path = f"{WORK_DIR}/{filename}_tokenized.jsonl"

    records = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:
            records.append(json.loads(line))

    return records


train_records = load_records("train")

validation_records = load_records("validation")

test_records = load_records("test")

print("\nTokenized datasets loaded.")

print("Training:", len(train_records))
print("Validation:", len(validation_records))
print("Testing:", len(test_records))


# ================================================================
# 4. CUSTOM PYTORCH DATASET
# ================================================================

class MalwareDataset(Dataset):

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):

        record = self.records[index]

        return {
            "input_ids": record["input_ids"],
            "labels": record["labels"]
        }


train_dataset = MalwareDataset(train_records)

validation_dataset = MalwareDataset(validation_records)

test_dataset = MalwareDataset(test_records)

print("\nPyTorch datasets created.")


# ================================================================
# 5. CUSTOM DYNAMIC PADDING COLLATOR
# ================================================================

def masked_collator(features):

    batch_size = len(features)

    max_len = max(
        len(x["input_ids"])
        for x in features
    )

    input_ids = torch.full(
        (batch_size, max_len),
        tokenizer.pad_token_id,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    for i, feature in enumerate(features):

        length = len(feature["input_ids"])

        input_ids[i, :length] = torch.tensor(
            feature["input_ids"],
            dtype=torch.long
        )

        attention_mask[i, :length] = 1

        labels[i, :length] = torch.tensor(
            feature["labels"],
            dtype=torch.long
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


# ================================================================
# 6. CREATE DATALOADERS
# ================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    collate_fn=masked_collator,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    collate_fn=masked_collator,
    num_workers=0
)

print("\nDataLoaders created.")

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))


# ================================================================
# 7. VERIFY MASKED BATCH
# ================================================================

sample_batch = next(iter(train_loader))

print("\nBatch verification:")

print("Input shape:", sample_batch["input_ids"].shape)

print("Attention shape:", sample_batch["attention_mask"].shape)

print("Labels shape:", sample_batch["labels"].shape)

print(
    "Masked tokens:",
    (sample_batch["labels"] == -100).sum().item()
)

print(
    "Supervised tokens:",
    (sample_batch["labels"] != -100).sum().item()
)

assert (
    sample_batch["labels"] != -100
).sum().item() > 0


# ================================================================
# 8. CONFIGURE 4-BIT NF4 QUANTIZATION
# ================================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("\n4-bit NF4 quantization configured.")


# ================================================================
# 9. LOAD DEEPSEEK-CODER
# ================================================================

print("\nLoading DeepSeek-Coder-1.3B...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.config.use_cache = False

print("DeepSeek model loaded.")


# ================================================================
# 10. PREPARE MODEL FOR QLoRA
# ================================================================

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("Model prepared for QLoRA.")


# ================================================================
# 11. CONFIGURE LoRA
# ================================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()


# ================================================================
# 12. TRAINING CONFIGURATION
# ================================================================

EPOCHS = 1

LEARNING_RATE = 2e-4

GRADIENT_ACCUMULATION_STEPS = 16

SAVE_STEPS = 200

LOGGING_STEPS = 10

print("\nTraining configuration:")

print("Epochs:", EPOCHS)

print("Learning rate:", LEARNING_RATE)

print("Micro batch size:", MICRO_BATCH_SIZE)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch size on one GPU:",
    MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
)

print("Checkpoint interval:", SAVE_STEPS)


# ================================================================
# 13. GPU MEMORY CHECK
# ================================================================

torch.cuda.empty_cache()
gc.collect()

for i in range(torch.cuda.device_count()):

    print(
        f"\nGPU {i}:",
        torch.cuda.get_device_name(i)
    )

    print(
        "Allocated:",
        round(
            torch.cuda.memory_allocated(i) / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Reserved:",
        round(
            torch.cuda.memory_reserved(i) / 1024**3,
            2
        ),
        "GB"
    )


# ================================================================
# 14. SAVE PREPARATION CONFIGURATION
# ================================================================

CONFIG = {
    "model": MODEL_NAME,
    "dataset": "Malware_Dataset.csv",
    "train_samples": len(train_dataset),
    "validation_samples": len(validation_dataset),
    "test_samples": len(test_dataset),
    "max_length": MAX_LENGTH,
    "quantization": "4-bit NF4",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "save_steps": SAVE_STEPS
}

with open(
    f"{MODEL_OUTPUT}/training_config.json",
    "w"
) as f:

    json.dump(CONFIG, f, indent=4)


# ================================================================
# FINAL STATUS
# ================================================================

print("\n" + "=" * 65)
print("CELL 2 COMPLETED")
print("=" * 65)

print("Dataset pipeline: READY")
print("DeepSeek base model: LOADED")
print("4-bit quantization: CONFIGURED")
print("LoRA: CONFIGURED")
print("Masked SFT: VERIFIED")

print("\nReady for Cell 3: Fine-tuning.")

GPU: Tesla T4
Available GPUs: 2

Tokenizer loaded.

Tokenized datasets loaded.
Training: 17693
Validation: 5898
Testing: 5898

PyTorch datasets created.

DataLoaders created.
Training batches: 17693
Validation batches: 5898

Batch verification:
Input shape: torch.Size([1, 2790])
Attention shape: torch.Size([1, 2790])
Labels shape: torch.Size([1, 2790])
Masked tokens: 2778
Supervised tokens: 12

4-bit NF4 quantization configured.

Loading DeepSeek-Coder-1.3B...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.69GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.69GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

DeepSeek model loaded.
Model prepared for QLoRA.
trainable params: 14,991,360 || all params: 1,361,463,296 || trainable%: 1.1011

Training configuration:
Epochs: 1
Learning rate: 0.0002
Micro batch size: 1
Gradient accumulation: 16
Effective batch size on one GPU: 16
Checkpoint interval: 200

GPU 0: Tesla T4
Allocated: 1.15 GB
Reserved: 1.42 GB

GPU 1: Tesla T4
Allocated: 0.0 GB
Reserved: 0.0 GB

CELL 2 COMPLETED
Dataset pipeline: READY
DeepSeek base model: LOADED
4-bit quantization: CONFIGURED
LoRA: CONFIGURED
Masked SFT: VERIFIED

Ready for Cell 3: Fine-tuning.


In [9]:
# ================================================================
# CELL 3: DEEPSEEK-CODER QLoRA DISTRIBUTED FINE-TUNING
# ================================================================

import os
import gc
import json
import subprocess
import sys
from pathlib import Path

import torch

WORK_DIR = Path("/kaggle/working/deepseek_malware")
OUTPUT_DIR = Path("/kaggle/working/deepseek_malware_final")
SCRIPT_PATH = Path("/kaggle/working/train_deepseek_ddp.py")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Release Cell 2's single-GPU model before launching DDP.
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print("Released Cell 2 model.")
print("Available GPUs:", torch.cuda.device_count())


# ------------------------------------------------
# WRITE DISTRIBUTED TRAINING SCRIPT
# ------------------------------------------------

training_script = r'''
import os
import gc
import json
import math
import shutil
from pathlib import Path

import torch
import torch.distributed as dist

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

WORK_DIR = Path("/kaggle/working/deepseek_malware")
OUTPUT_DIR = Path("/kaggle/working/deepseek_malware_final")

MAX_LENGTH = 4096

MICRO_BATCH_SIZE = 1
GRAD_ACCUMULATION = 16

EPOCHS = 1
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3

WARMUP_RATIO = 0.03

os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ============================================================
# DISTRIBUTED INITIALIZATION
# ============================================================

dist.init_process_group(backend="nccl")

LOCAL_RANK = int(os.environ["LOCAL_RANK"])
RANK = dist.get_rank()
WORLD_SIZE = dist.get_world_size()

torch.cuda.set_device(LOCAL_RANK)
DEVICE = torch.device(f"cuda:{LOCAL_RANK}")

def log(*args):
    if RANK == 0:
        print(*args, flush=True)

log("=" * 65)
log("DEEPSEEK DISTRIBUTED FINE-TUNING")
log("=" * 65)
log("World size:", WORLD_SIZE)
log("Current GPU:", torch.cuda.get_device_name(LOCAL_RANK))


# ============================================================
# LOAD TOKENIZED DATA
# ============================================================

class MalwareDataset(Dataset):

    def __init__(self, file_path):
        self.records = []

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                self.records.append(json.loads(line))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]

        return {
            "input_ids": torch.tensor(
                item["input_ids"], dtype=torch.long
            ),
            "labels": torch.tensor(
                item["labels"], dtype=torch.long
            )
        }


def collate_fn(batch):

    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = torch.full(
        (len(batch), max_len),
        tokenizer.pad_token_id,
        dtype=torch.long
    )

    labels = torch.full(
        (len(batch), max_len),
        -100,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (len(batch), max_len),
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        length = len(item["input_ids"])

        input_ids[i, :length] = item["input_ids"]
        labels[i, :length] = item["labels"]
        attention_mask[i, :length] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = MalwareDataset(
    WORK_DIR / "train_tokenized.jsonl"
)

validation_dataset = MalwareDataset(
    WORK_DIR / "validation_tokenized.jsonl"
)

train_sampler = DistributedSampler(
    train_dataset,
    num_replicas=WORLD_SIZE,
    rank=RANK,
    shuffle=True,
    seed=42
)

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    sampler=train_sampler,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

log("Training examples:", len(train_dataset))
log("Validation examples:", len(validation_dataset))
log("Batches per GPU:", len(train_loader))


# ============================================================
# LOAD 4-BIT MODEL
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

log(f"Loading model on GPU {LOCAL_RANK}...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": LOCAL_RANK},
    dtype=torch.float16,
    trust_remote_code=True
)

base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(base_model, lora_config)

model = model.to(DEVICE)

model = DDP(
    model,
    device_ids=[LOCAL_RANK],
    output_device=LOCAL_RANK,
    find_unused_parameters=False
)

if RANK == 0:
    model.module.print_trainable_parameters()

log(f"Model ready on GPU {LOCAL_RANK}.")


# ============================================================
# OPTIMIZER AND SCHEDULER
# ============================================================

trainable_parameters = [
    p for p in model.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUMULATION
)

total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

def lr_lambda(current_step):

    if current_step < warmup_steps:
        return float(current_step + 1) / float(warmup_steps)

    remaining = max(0, total_steps - current_step)

    return remaining / max(1, total_steps - warmup_steps)


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

log("Optimizer steps per epoch:", steps_per_epoch)
log("Total optimizer steps:", total_steps)
log("Warmup steps:", warmup_steps)


# ============================================================
# CHECKPOINT MANAGEMENT
# ============================================================

checkpoint_dirs = []

def save_checkpoint(step):

    dist.barrier()

    if RANK == 0:

        checkpoint_dir = (
            OUTPUT_DIR / f"checkpoint-{step}"
        )

        checkpoint_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        model.module.save_pretrained(
            checkpoint_dir,
            safe_serialization=True
        )

        tokenizer.save_pretrained(checkpoint_dir)

        torch.save(
            {
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "global_step": step
            },
            checkpoint_dir / "training_state.pt"
        )

        checkpoint_dirs.append(checkpoint_dir)

        while len(checkpoint_dirs) > SAVE_TOTAL_LIMIT:

            old_checkpoint = checkpoint_dirs.pop(0)

            shutil.rmtree(
                old_checkpoint,
                ignore_errors=True
            )

        print(
            f"\nCHECKPOINT SAVED: {checkpoint_dir}",
            flush=True
        )

        print(
            f"Retained checkpoints: {len(checkpoint_dirs)}",
            flush=True
        )

    dist.barrier()


# ============================================================
# TRAINING LOOP
# ============================================================

global_step = 0
optimizer.zero_grad(set_to_none=True)

training_losses = []

for epoch in range(EPOCHS):

    model.train()
    train_sampler.set_epoch(epoch)

    running_loss = 0.0
    accumulated_loss = 0.0

    total_batches = len(train_loader)

    log(f"\nStarting epoch {epoch + 1}/{EPOCHS}")

    for batch_idx, batch in enumerate(train_loader):

        group_start = (
            batch_idx // GRAD_ACCUMULATION
        ) * GRAD_ACCUMULATION

        group_size = min(
            GRAD_ACCUMULATION,
            total_batches - group_start
        )

        is_update_step = (
            (batch_idx + 1) % GRAD_ACCUMULATION == 0
            or batch_idx + 1 == total_batches
        )

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        sync_context = (
            model.no_sync()
            if not is_update_step
            else torch.enable_grad()
        )

        with sync_context:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss

            scaled_loss = loss / group_size
            scaled_loss.backward()

        running_loss += loss.detach().item()
        accumulated_loss += loss.detach().item()

        if is_update_step:

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            average_loss = accumulated_loss / group_size
            accumulated_loss = 0.0

            training_losses.append({
                "step": global_step,
                "loss": average_loss,
                "learning_rate": scheduler.get_last_lr()[0]
            })

            if global_step % 10 == 0:

                log(
                    f"Step {global_step}/{total_steps} | "
                    f"Loss: {average_loss:.5f} | "
                    f"LR: {scheduler.get_last_lr()[0]:.7f}"
                )

            if global_step % SAVE_STEPS == 0:
                save_checkpoint(global_step)

    epoch_loss = running_loss / total_batches

    log(
        f"\nEpoch {epoch + 1} completed. "
        f"Average training loss: {epoch_loss:.5f}"
    )


# ============================================================
# FINAL MODEL SAVE
# ============================================================

save_checkpoint(global_step)

dist.barrier()

if RANK == 0:

    model.module.save_pretrained(
        OUTPUT_DIR,
        safe_serialization=True
    )

    tokenizer.save_pretrained(OUTPUT_DIR)

    with open(
        OUTPUT_DIR / "training_losses.json",
        "w"
    ) as f:

        json.dump(training_losses, f, indent=2)

    training_summary = {
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "global_steps": global_step,
        "training_examples": len(train_dataset),
        "validation_examples": len(validation_dataset),
        "learning_rate": LEARNING_RATE,
        "gradient_accumulation": GRAD_ACCUMULATION,
        "world_size": WORLD_SIZE,
        "checkpoint_interval": SAVE_STEPS,
        "checkpoint_retention": SAVE_TOTAL_LIMIT
    }

    with open(
        OUTPUT_DIR / "training_summary.json",
        "w"
    ) as f:

        json.dump(training_summary, f, indent=2)

    print("\n" + "=" * 65)
    print("FINE-TUNING COMPLETED")
    print("=" * 65)
    print("Final model:", OUTPUT_DIR)
    print("Optimizer steps:", global_step)
    print("Checkpoints retained:", len(checkpoint_dirs))
    print("Training summary saved.")
    print("=" * 65)


dist.barrier()
dist.destroy_process_group()
'''


SCRIPT_PATH.write_text(training_script)

print("Distributed training script created.")


# ------------------------------------------------
# LAUNCH TWO-GPU TRAINING
# ------------------------------------------------

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["NCCL_DEBUG"] = "WARN"

command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nnodes=1",
    "--nproc_per_node=2",
    str(SCRIPT_PATH)
]

print("\nLaunching training on both T4 GPUs...")
print("Checkpoint interval: 50 optimizer steps")
print("Maximum retained checkpoints: 3\n")

result = subprocess.run(
    command,
    env=env
)

if result.returncode != 0:
    raise RuntimeError(
        "Distributed training failed. Review the error output above."
    )

print("\nCELL 3 COMPLETED SUCCESSFULLY.")
print("Final adapter:", OUTPUT_DIR)

Released Cell 2 model.
Available GPUs: 2
Distributed training script created.

Launching training on both T4 GPUs...
Checkpoint interval: 50 optimizer steps
Maximum retained checkpoints: 3




*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


DEEPSEEK DISTRIBUTED FINE-TUNING
World size: 2
Current GPU: Tesla T4
Training examples: 17693
Validation examples: 5898
Batches per GPU: 8847
Loading model on GPU 0...


Loading weights: 100%|██████████| 219/219 [00:02<00:00, 78.72it/s] 


NCCL version 2.32.3+cuda12.9

[2026-11-11 15:57:44] 9801beb21fd4:279:279 [1] transport/rma_socket.cc:32 (ncclRmaSocketProxyInit) NCCL WARN RMA/Socket : rma_socket requires GDRCopy; and GDR Copy is not available

[2026-11-11 15:57:44] 9801beb21fd4:278:278 [0] transport/rma_socket.cc:32 (ncclRmaSocketProxyInit) NCCL WARN RMA/Socket : rma_socket requires GDRCopy; and GDR Copy is not available
trainable params: 14,991,360 || all params: 1,361,463,296 || trainable%: 1.1011
Model ready on GPU 0.
Optimizer steps per epoch: 553
Total optimizer steps: 553
Warmup steps: 16

Starting epoch 1/1
Step 10/553 | Loss: 0.49082 | LR: 0.0001375
Step 20/553 | Loss: 0.18441 | LR: 0.0001985
Step 30/553 | Loss: 0.19486 | LR: 0.0001948
Step 40/553 | Loss: 0.21468 | LR: 0.0001911
Step 50/553 | Loss: 0.21184 | LR: 0.0001873


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)



CHECKPOINT SAVED: /kaggle/working/deepseek_malware_final/checkpoint-50
Retained checkpoints: 1
Step 60/553 | Loss: 0.17414 | LR: 0.0001836
Step 70/553 | Loss: 0.16115 | LR: 0.0001799
Step 80/553 | Loss: 0.16892 | LR: 0.0001762
Step 90/553 | Loss: 0.17047 | LR: 0.0001724
Step 100/553 | Loss: 0.15184 | LR: 0.0001687

CHECKPOINT SAVED: /kaggle/working/deepseek_malware_final/checkpoint-100
Retained checkpoints: 2
Step 110/553 | Loss: 0.14301 | LR: 0.0001650
Step 120/553 | Loss: 0.15763 | LR: 0.0001613
Step 130/553 | Loss: 0.20781 | LR: 0.0001575
Step 140/553 | Loss: 0.14072 | LR: 0.0001538
Step 150/553 | Loss: 0.11118 | LR: 0.0001501

CHECKPOINT SAVED: /kaggle/working/deepseek_malware_final/checkpoint-150
Retained checkpoints: 3
Step 160/553 | Loss: 0.09950 | LR: 0.0001464
Step 170/553 | Loss: 0.11413 | LR: 0.0001426
Step 180/553 | Loss: 0.14347 | LR: 0.0001389
Step 190/553 | Loss: 0.05903 | LR: 0.0001352
Step 200/553 | Loss: 0.15813 | LR: 0.0001315

CHECKPOINT SAVED: /kaggle/working/deep